In [2]:
import cv2
import os

# =========================
# Funciones
# =========================
def encontrar_esquinas(imagen_gris):
    detector = cv2.FastFeatureDetector_create(
        threshold=60,
        nonmaxSuppression=True,
        type=cv2.FastFeatureDetector_TYPE_7_12
    )
    puntos = detector.detect(imagen_gris, mask=None)
    return puntos


def dibujar_esquinas(imagen, puntos):
    return cv2.drawKeypoints(imagen, puntos, None, color=(0, 0, 255))


def calcular_densidad(imagen_gris):
    pixeles_activos = (imagen_gris > 0).sum()
    total_pixeles = imagen_gris.size
    return (pixeles_activos / total_pixeles) * 100


def procesar_imagen(ruta_imagen):
    img = cv2.imread(ruta_imagen)
    
    if img is None:
        return None
    
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detectar esquinas
    puntos = encontrar_esquinas(gris)
    
    # Dibujar esquinas
    img_con_puntos = dibujar_esquinas(img.copy(), puntos)
    
    # Calcular densidad
    densidad = calcular_densidad(gris)
    
    return img, img_con_puntos, densidad, len(puntos)


# =========================
# Programa principal
# =========================
ruta = "img/telas/"
archivos = os.listdir(ruta)

resultados = []

for archivo in archivos:
    if not archivo.lower().endswith((".jpg", ".png", ".jpeg")):
        continue
    
    ruta_completa = os.path.join(ruta, archivo)
    
    resultado = procesar_imagen(ruta_completa)
    
    if resultado is None:
        continue
    
    img, img_pts, densidad, num_esquinas = resultado
    
    resultados.append([archivo, densidad, num_esquinas])
    
    print("Procesando:", archivo)
    print("Esquinas:", num_esquinas)
    
    # Mostrar comparación
    comparacion = cv2.hconcat([img, img_pts])
    cv2.imshow("Comparacion de telas", comparacion)
    cv2.waitKey(3000)

cv2.destroyAllWindows()
 
# =========================
# Resultados finales
# =========================
print("\nTabla de resultados:")
for r in resultados:
    print(r)

Procesando: Corduroy.jpg
Esquinas: 3


Procesando: Cotton-Flannel.jpg
Esquinas: 1
Procesando: Cotton-Jersey.jpg
Esquinas: 1
Procesando: Faux-Fur.jpg
Esquinas: 24
Procesando: Felt-Fabric.jpg
Esquinas: 0
Procesando: Linen.jpg
Esquinas: 8
Procesando: Oxford.jpg
Esquinas: 11650
Procesando: Polyester.jpg
Esquinas: 2378
Procesando: Satin.jpg
Esquinas: 0
Procesando: Sherpa.jpg
Esquinas: 0
Procesando: Silk.jpg
Esquinas: 215
Procesando: Terry.jpg
Esquinas: 3369
Procesando: Tweed.jpg
Esquinas: 281
Procesando: Velvet.jpg
Esquinas: 0

Tabla de resultados:
['Corduroy.jpg', np.float64(100.0), 3]
['Cotton-Flannel.jpg', np.float64(100.0), 1]
['Cotton-Jersey.jpg', np.float64(100.0), 1]
['Faux-Fur.jpg', np.float64(100.0), 24]
['Felt-Fabric.jpg', np.float64(100.0), 0]
['Linen.jpg', np.float64(100.0), 8]
['Oxford.jpg', np.float64(100.0), 11650]
['Polyester.jpg', np.float64(100.0), 2378]
['Satin.jpg', np.float64(100.0), 0]
['Sherpa.jpg', np.float64(100.0), 0]
['Silk.jpg', np.float64(100.0), 215]
['Terry.jpg', np.float64(100.0), 3369]
['Tweed.jp

In [6]:
import cv2
import numpy as np

# =========================
# Funciones
# =========================
def cargar_imagenes(ruta1, ruta2):
    img1 = cv2.imread(ruta1)
    img2 = cv2.imread(ruta2)
    return img1, img2


def convertir_gris(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def detectar_orb(gray):
    orb = cv2.ORB_create()
    kp, des = orb.detectAndCompute(gray, None)
    return kp, des


def emparejar(des1, des2):
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    return sorted(matches, key=lambda x: x.distance)


def obtener_puntos(kp1, kp2, matches, n=3):
    pts1, pts2 = [], []
    
    for m in matches[:n]:
        pts1.append(kp1[m.queryIdx].pt)
        pts2.append(kp2[m.trainIdx].pt)
    
    return np.float32(pts1), np.float32(pts2)


def transformar_puntos(M, pts):
    pts_hom = np.hstack([pts, np.ones((pts.shape[0], 1))])
    pts_transf = np.dot(M, pts_hom.T).T
    return pts_transf


def dibujar_puntos(img, originales, transformados):
    salida = img.copy()
    
    for p in originales:
        cv2.circle(salida, tuple(np.int32(p)), 5, (0, 0, 0), -1)
    
    for p in transformados:
        cv2.circle(salida, tuple(np.int32(p)), 5, (0, 0, 0), -1)
    
    return salida

# =========================
# Programa principal
# =========================
img1, img2 = cargar_imagenes("img/lena.jpeg", "img/lena_rotada.jpeg")

gray1 = convertir_gris(img1)
gray2 = convertir_gris(img2)

kp1, des1 = detectar_orb(gray1)
kp2, des2 = detectar_orb(gray2)

matches = emparejar(des1, des2)

# Obtener 3 mejores puntos
pts1, pts2 = obtener_puntos(kp1, kp2, matches, 3)

# Transformación afín
M = cv2.getAffineTransform(pts1, pts2)

# Transformar puntos
pts_transformados = transformar_puntos(M, pts1)

# Mostrar en consola
print("\nPuntos originales:\n", pts1)
print("\nPuntos transformados:\n", pts_transformados)
print("\nPuntos destino:\n", pts2)
print("\nMatriz afín:\n", M)

# Aplicar transformación a la imagen
filas, cols = img1.shape[:2]
resultado = cv2.warpAffine(img1, M, (cols, filas))

# Dibujar puntos
img_puntos = dibujar_puntos(img1, pts1, pts_transformados)

# =========================
# Visualización
# =========================
cv2.imshow("Puntos final ", img_puntos)

comparacion = cv2.hconcat([img1, img2, resultado])
cv2.imshow("Comparacion", comparacion)

img_matches = cv2.drawMatches(img1, kp1, img2, kp2, matches[:3], None)
cv2.imshow("Matches", img_matches)

cv2.waitKey(0)
cv2.destroyAllWindows()


Puntos originales:
 [[162.       92.     ]
 [141.      116.     ]
 [ 85.01761  95.38561]]

Puntos transformados:
 [[107.          59.        ]
 [124.          86.        ]
 [ 89.16481018 134.78401184]]

Puntos destino:
 [[107.       59.     ]
 [124.       86.     ]
 [ 89.16481 134.78401]]

Matriz afín:
 [[  0.2733496    0.94751423 -24.45394452]
 [ -0.97237526   0.27417165 191.3010008 ]]


In [9]:
import cv2
import numpy as np

# =========================
# Parámetros
# =========================
lk_params = dict(
    winSize=(15, 15),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.01)
)

feature_params = dict(
    maxCorners=200,
    qualityLevel=0.2,
    minDistance=5,
    blockSize=7
)

MAX_ERROR = 20
MAX_DISPLACEMENT = 50
MIN_POINTS_RATIO = 0.2

# =========================
# Gamma
# =========================
def adjust_gamma(image, gamma=1.5):
    invGamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** invGamma * 255
                      for i in np.arange(256)]).astype("uint8")
    return cv2.LUT(image, table)

# =========================
# Funciones
# =========================
def detect_features(gray, mask=None):
    pts = cv2.goodFeaturesToTrack(gray, mask=mask, **feature_params)
    if pts is not None:
        pts = np.float32(pts)
    return pts

def filter_points(p0, p1, st, err):
    if p1 is None or st is None or err is None:
        return None, None

    st = st.reshape(-1)
    err = err.reshape(-1)

    valid = (st == 1) & (err < MAX_ERROR)

    p0_valid = p0[valid]
    p1_valid = p1[valid]

    displacement = np.linalg.norm(p1_valid - p0_valid, axis=2).reshape(-1)
    drift_mask = displacement < MAX_DISPLACEMENT

    return p0_valid[drift_mask], p1_valid[drift_mask]

# =========================
# Video
# =========================
video = cv2.VideoCapture("img/Traffic.mp4")

ret, old_frame = video.read()
if not ret:
    raise Exception("No se pudo leer el video")

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
old_gray = adjust_gamma(old_gray, gamma=1.5)
old_gray = cv2.GaussianBlur(old_gray, (5,5), 0)

# =========================
# Máscara centrada
# =========================
corner_mask = np.zeros_like(old_gray, dtype=np.uint8)
h, w = old_gray.shape

x1 = int(w * 0.4)
x2 = int(w * 0.63)
y1 = int(h * 0.38)
y2 = int(h * 0.63)

corner_mask[y1:y2, x1:x2] = 255

p0 = detect_features(old_gray, corner_mask)

mask_draw = np.zeros_like(old_frame)

print("Presiona 'q' para salir")

while True:
    ret, frame = video.read()

    if not ret:
        print("Reiniciando video...")
        video.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    # Preprocesamiento
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_gray = adjust_gamma(frame_gray, gamma=3)
    frame_gray = cv2.GaussianBlur(frame_gray, (5,5), 0)

    # Optical Flow
    if p0 is not None and len(p0) > 0:
        p1, st, err = cv2.calcOpticalFlowPyrLK(
            old_gray, frame_gray, p0, None, **lk_params
        )

        good_old, good_new = filter_points(p0, p1, st, err)

        if good_new is not None and len(good_new) > 0:

            for new, old in zip(good_new, good_old):
                a, b = new.ravel()
                c, d = old.ravel()

                # Línea negra
                mask_draw = cv2.line(mask_draw, (int(c), int(d)),
                                     (int(a), int(b)),
                                     (0, 0, 0), 2)

                # Punto negro
                frame = cv2.circle(frame, (int(a), int(b)), 4,
                                   (0, 0, 0), -1)

            p0 = good_new.reshape(-1, 1, 2)

        else:
            p0 = None

    # Re-detección
    if p0 is None or len(p0) < feature_params['maxCorners'] * MIN_POINTS_RATIO:
        print("Re-detectando puntos...")

        new_pts = detect_features(frame_gray, corner_mask)

        if new_pts is not None:
            if p0 is not None:
                p0 = np.concatenate((p0, new_pts), axis=0)
            else:
                p0 = new_pts

        mask_draw = np.zeros_like(frame)

    # Visualización
    output = cv2.add(frame, mask_draw)

    cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv2.putText(output, f"Puntos activos: {0 if p0 is None else len(p0)}",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 255, 0), 2)

    cv2.imshow("Optical Flow (Robusto)", output)

    old_gray = frame_gray.copy()

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()

Presiona 'q' para salir
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
Re-detectando puntos...
